If you want to type along with me, head to [this notebook](https://humboldt.cloudbank.2i2c.cloud/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2Fmlekimchi%2Fdata111_fall26&branch=main&urlpath=tree%2Fdata111_fall26%2Flectures%2Flec07_live.ipynb) instead. If you prefer follow along by executing the cells, stay in this notebook.

# Lecture 7: Visualizing Data — Line, Scatter, Bar
v.ekc

A table shows you numbers; a chart shows you the *story*. Today we learn the three workhorse charts and — just as important — **which one to use when**. (Slides: KL Lecture 07 deck.)

**Today's sections:**
1. Census recap
2. Line plots
3. Analysis: males vs. females
4. Scatter plots
5. Bar charts

In [ ]:
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

---
## 1. Census Recap

Same census table as last time — rebuilt in three tidy steps. (Remember the codes: `SEX` 0/1/2 = total/male/female; `AGE` 999 = all ages.)

In [ ]:
full = Table.read_table('nc-est2019-agesex-res.csv')

In [ ]:
# Keep only the columns we care about
partial = full.select('SEX', 'AGE', 'POPESTIMATE2019')

In [ ]:
# Make things easier to read
us_pop_2019 = partial.relabeled(2, '2019')
us_pop_2019

In [ ]:
us_pop_2019.where('AGE', are.equal_to(999))

In [ ]:
us_pop_2019.where('AGE', are.equal_to(0))

In [ ]:
us_pop_2019.where('AGE', are.equal_to(100))

---
## 2. Line Plots

**Line plots** are for numerical values in a natural order — like age, or time. (KL deck, slides 16–19.)

### 📋 Board Reference

| Code | What it does |
|---|---|
| `t.plot('x', 'y')` | line plot of y against x |
| `t.plot('x')` | one line per remaining column |
| `t.scatter('x', 'y')` | scatter plot — two numerical variables |
| `t.barh('category', 'value')` | horizontal bar chart |
| `t.set_format('col', PercentFormatter)` | display a column as percents |
| `t.take(np.arange(a, b))` | rows a through b−1 |

In [ ]:
# Remove the age totals
no_999 = us_pop_2019.where('AGE', are.below(999))

In [ ]:
overall = no_999.where('SEX', 0).drop('SEX')
overall

In [ ]:
## Our first line plot! 
overall.plot('AGE', '2019')

**Options for labeling**

In [ ]:
### OPTION 1
# US Population  <--- Just add a comment
overall.plot('AGE', '2019')

In [ ]:
### OPTION 2
overall.plot('AGE', '2019')
print('US Population')  # <--- Print out what it is

In [ ]:
### OPTION 3
overall.plot('AGE', '2019')
plots.title('US Population');    # <--- not needed for Data 111, but a handy skill!

---
## 3. Analysis: Males vs. Females

One plot is a picture; comparing plots is an **analysis**. Your turn to set it up.

### Try It 1: Build the tables 😊

1. Create a table `males` with columns `AGE` and `2019` — the male population at every age (no totals!).

2. Same for `females`.

In [ ]:
# Fill in the blank — this cell errors until you do! 😅
males = 
males

In [ ]:
# Fill in the blank — this cell errors until you do! 😅
females = 
females

<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Filter `no_999` (totals already removed) by the SEX code, then drop the code column.*

```python
# 1.
males = no_999.where('SEX', 1).drop('SEX')

# 2.
females = no_999.where('SEX', 2).drop('SEX')
```

</details>

Run the next cell so the rest of the lecture works:

In [ ]:
# run me: needed below
males = no_999.where('SEX', 1).drop('SEX')
females = no_999.where('SEX', 2).drop('SEX')

In [ ]:
males_females_2019 = Table().with_columns(
    'Age', males.column('AGE'),
    'Males', males.column('2019'),
    'Females', females.column('2019')
)
males_females_2019

In [ ]:
males_females_2019.plot('Age')

In [ ]:
# Calculate the percent female for each age
total = males_females_2019.column('Males') + males_females_2019.column('Females')
proportion_female = males_females_2019.column('Females') / total 
proportion_female

In [ ]:
# Add female percent to our table
males_females_2019 = males_females_2019.with_column('Proportion female', proportion_female).set_format('Proportion female',PercentFormatter)
males_females_2019

In [ ]:
males_females_2019.plot('Age', 'Proportion female')

In [ ]:
# ^^ Look at the y-axis! Trend is not as dramatic as you might think

males_females_2019.plot('Age', 'Proportion female')
plots.ylim(0, 1);  # Optional for Data 111

> **Always check the y-axis!** 😬 The first plot looked dramatic; with the axis starting at 0, the story is much gentler. Same data, honest picture. (KL deck, slides 28–29.)

In [ ]:
males_females_2019.take(np.arange(35,45))

---
## 4. Scatter Plots

When *neither* variable has a natural order — like number of movies vs. total gross — use a **scatter plot**: every dot is one individual.

In [ ]:
# Actors and their highest grossing movies
actors = Table.read_table('actors.csv')
actors

In [ ]:
actors.scatter('Number of Movies', 'Total Gross')

In [ ]:
actors.scatter('Number of Movies', 'Average per Movie')

In [ ]:
actors.where('Average per Movie', are.above(400))

> **Line vs. scatter:** if the x-values have a natural sequence (age, year), a line plot works. If each row is just an individual, connecting the dots means nothing — scatter it. Guess who that outlier dot is, then check the table above. 🎬

---
## 5. Bar Charts

Categories (studios, titles, flavors) aren't numbers — they get **bar charts**. Watch what happens when we wrongly force a line plot on them:

In [ ]:
# Highest grossing movies as of 2017
top_movies = Table.read_table('top_movies_2017.csv')
top_movies

In [ ]:
top10_adjusted = top_movies.take(np.arange(10))
top10_adjusted

In [ ]:
# Convert to millions of dollars for readability
millions = np.round(top10_adjusted.column('Gross (Adjusted)') / 1000000, 3)
top10_adjusted = top10_adjusted.with_column('Millions', millions)
top10_adjusted

In [ ]:
# A line plot doesn't make sense here: don't do this!
top10_adjusted.plot('Year', 'Millions')

> **Don't do this!** A line plot over `Year` implies a sequence and interpolation that make no sense for 10 individual movies. (KL deck, slides 24–25.) Bar chart to the rescue:

In [ ]:
top10_adjusted.barh('Title', 'Millions')

In [ ]:
top10_adjusted = top10_adjusted.with_column('Age of Movie', 2024 - top10_adjusted.column('Year'))

In [ ]:
top10_adjusted.barh('Title','Age of Movie')

---
> **Today's takeaway:** line plots for ordered numbers, scatter plots for pairs of numbers, bar charts for categories — and never trust a plot until you've checked its y-axis. Homework 3 uses all three.

## Appendix — Quick Reference

Full `datascience` docs: [data8.org/datascience](https://data8.org/datascience/)

### Minimal template — the three charts
```python
t.plot('x', 'y')        # ordered numbers
t.scatter('x', 'y')     # two numerical variables
t.barh('category', 'value')   # categorical
```